# 04 - Feature Engineering

**Purpose:** Build lag, rolling-window, and calendar features for each dataset, fold-aware.

**Inputs:** data/interim/* (from notebook 02)

**Outputs:** Feature matrices under data/processed/.

**Runtime estimate:** 5-15 minutes.

## Lag features

In [1]:
import pandas as pd

from sdg_leakage_audit.features.lags import make_lag_features
from sdg_leakage_audit.utils.io import read_yaml
from sdg_leakage_audit.utils.paths import configs_dir, data_interim_dir, to_project_relative

datasets_cfg = read_yaml(configs_dir() / "datasets.yaml")["datasets"]

interim = {}  # dataset_id -> cleaned DataFrame (from notebook 02)
for dataset_id, cfg in datasets_cfg.items():
    interim[dataset_id] = pd.read_parquet(data_interim_dir() / f"{dataset_id}.parquet")

# Lag/rolling-window sizes are taken directly from each dataset's configured forecast
# horizons -- the exact steps-ahead the models will eventually need to predict, so these
# are also the most informative lookback distances to expose as features.
lag_demo = make_lag_features(
    interim["household_power_consumption"],
    column=datasets_cfg["household_power_consumption"]["target_column"],
    lags=datasets_cfg["household_power_consumption"]["forecast_horizons"],
)
print("Example lag features (household_power_consumption):")
print(lag_demo.filter(like="_lag").head())


Example lag features (household_power_consumption):
                     Global_active_power_lag1  Global_active_power_lag24  \
timestamp                                                                  
2006-12-16 17:24:00                       NaN                        NaN   
2006-12-16 17:25:00                     4.216                        NaN   
2006-12-16 17:26:00                     5.360                        NaN   
2006-12-16 17:27:00                     5.374                        NaN   
2006-12-16 17:28:00                     5.388                        NaN   

                     Global_active_power_lag60  
timestamp                                       
2006-12-16 17:24:00                        NaN  
2006-12-16 17:25:00                        NaN  
2006-12-16 17:26:00                        NaN  
2006-12-16 17:27:00                        NaN  
2006-12-16 17:28:00                        NaN  


## Rolling-window features

In [2]:
from sdg_leakage_audit.features.rolling import make_rolling_features

rolling_demo = make_rolling_features(
    interim["household_power_consumption"],
    column=datasets_cfg["household_power_consumption"]["target_column"],
    windows=datasets_cfg["household_power_consumption"]["forecast_horizons"],
    stats=["mean", "std"],
)
print("Example rolling features (household_power_consumption):")
print(rolling_demo.filter(like="_roll").head(10))


Example rolling features (household_power_consumption):
                     Global_active_power_roll1_mean  \
timestamp                                             
2006-12-16 17:24:00                           4.216   
2006-12-16 17:25:00                           5.360   
2006-12-16 17:26:00                           5.374   
2006-12-16 17:27:00                           5.388   
2006-12-16 17:28:00                           3.666   
2006-12-16 17:29:00                           3.520   
2006-12-16 17:30:00                           3.702   
2006-12-16 17:31:00                           3.700   
2006-12-16 17:32:00                           3.668   
2006-12-16 17:33:00                           3.662   

                     Global_active_power_roll1_std  \
timestamp                                            
2006-12-16 17:24:00                            NaN   
2006-12-16 17:25:00                            NaN   
2006-12-16 17:26:00                            NaN   
2006-12-16 17

## Calendar features

In [3]:
from sdg_leakage_audit.features.calendar import make_calendar_features

calendar_demo = make_calendar_features(interim["household_power_consumption"])
print("Example calendar features (household_power_consumption):")
print(calendar_demo.filter(like="hour").join(calendar_demo.filter(like="month")).head())


Example calendar features (household_power_consumption):
                     hour  hour_sin  hour_cos  month     month_sin  month_cos
timestamp                                                                    
2006-12-16 17:24:00    17 -0.965926 -0.258819     12 -2.449294e-16        1.0
2006-12-16 17:25:00    17 -0.965926 -0.258819     12 -2.449294e-16        1.0
2006-12-16 17:26:00    17 -0.965926 -0.258819     12 -2.449294e-16        1.0
2006-12-16 17:27:00    17 -0.965926 -0.258819     12 -2.449294e-16        1.0
2006-12-16 17:28:00    17 -0.965926 -0.258819     12 -2.449294e-16        1.0


## Fold-aware feature build pipeline

In [4]:
from sdg_leakage_audit.features.build import build_feature_matrix
from sdg_leakage_audit.utils.paths import data_processed_dir

data_processed_dir().mkdir(parents=True, exist_ok=True)

feature_summary_rows = []
for dataset_id, cfg in datasets_cfg.items():
    df = interim[dataset_id]
    horizons = cfg["forecast_horizons"]

    if dataset_id == "beijing_multisite_air_quality":
        # Lag/rolling features must never cross a station boundary -- each station is an
        # independent series sharing the same timestamp range, so build per-group and
        # concatenate, exactly mirroring how clean_and_report (notebook 02) grouped by station.
        parts = []
        for station, group in df.groupby("station"):
            feats = build_feature_matrix(
                group.drop(columns=["station"]),
                target_column=cfg["target_column"],
                lags=horizons,
                rolling_windows=horizons,
            )
            feats["station"] = station
            parts.append(feats)
        features = pd.concat(parts).sort_index()
    else:
        features = build_feature_matrix(
            df,
            target_column=cfg["target_column"],
            lags=horizons,
            rolling_windows=horizons,
        )

    out_path = data_processed_dir() / f"{dataset_id}_features.parquet"
    features.to_parquet(out_path)

    n_feature_cols = features.shape[1] - df.shape[1]
    pct_rows_with_full_history = float((~features.filter(like="_lag").isna().any(axis=1)).mean() * 100.0)
    feature_summary_rows.append(
        {
            "dataset_id": dataset_id,
            "n_rows": len(features),
            "n_feature_columns": n_feature_cols,
            "pct_rows_with_full_lag_history": pct_rows_with_full_history,
            "saved_to": str(to_project_relative(out_path)),
        }
    )
    print(f"{dataset_id:32s} -> {to_project_relative(out_path)} ({features.shape[1]} columns)")

feature_summary_df = pd.DataFrame(feature_summary_rows).set_index("dataset_id")
feature_summary_df


household_power_consumption      -> data\processed\household_power_consumption_features.parquet (26 columns)
appliances_energy_prediction     -> data\processed\appliances_energy_prediction_features.parquet (47 columns)
air_quality_uci                  -> data\processed\air_quality_uci_features.parquet (32 columns)


beijing_multisite_air_quality    -> data\processed\beijing_multisite_air_quality_features.parquet (32 columns)


jena_climate                     -> data\processed\jena_climate_features.parquet (33 columns)


,n_rows,n_feature_columns,pct_rows_with_full_lag_history,saved_to
dataset_id,,,,
household_power_consumption,2075259,19,98.698861,data\processed\household_power_consumption_fea...
appliances_energy_prediction,19735,19,99.878389,data\processed\appliances_energy_prediction_fe...
air_quality_uci,9357,19,72.694240,data\processed\air_quality_uci_features.parquet
beijing_multisite_air_quality,420768,19,95.246549,data\processed\beijing_multisite_air_quality_f...
jena_climate,420768,19,99.776837,data\processed\jena_climate_features.parquet


## Findings / what to carry forward

- Lag and rolling-window features use each dataset's own configured `forecast_horizons`
  as the lookback distances, so the feature set is directly tied to what the models will
  later need to predict (notebooks 06-07).
- `build_feature_matrix` never drops or imputes rows -- the earliest rows of each fold have
  NaN lag/rolling features by construction (insufficient history). This is intentional: the
  protocol splitters (notebook 05) and model training (notebook 06) decide how to handle
  that NaN region, since dropping it must happen fold-locally to stay leakage-safe.
- Beijing multi-site air quality features were built per-station (grouped), never across
  station boundaries -- a lag computed across two different stations' concatenated rows
  would silently reference an unrelated station's past value.
- Feature matrices saved to `data/processed/<dataset_id>_features.parquet`, ready for
  notebook 05's protocol/split construction and notebook 06's P0 baseline models.
- All lag/rolling features here are the CAUSAL variants (matching
  `protocols.injectors.lag_feature_correct_train_fold_only` /
  `rolling_feature_correct_causal`). The leaky P4 variants (centered rolling,
  full-series-before-split lag) are deliberately NOT used here -- they are only
  constructed later, per-protocol, inside notebook 07's injection grid.
